4. Write a drift check: KS-test each feature's last-7-days distribution vs the training baseline; print a red/green
report. Simulate drift by shifting days_since_last_order +20 and confirm the alarm fires.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

In [ ]:
orders = pd.read_csv("churn_features.csv")

In [12]:
#feature engineering
orders["first_order_date"] = pd.to_datetime(orders["first_order_date"])
orders["last_order_date"] = pd.to_datetime(orders["last_order_date"])
today = orders["last_order_date"].max()
features = pd.DataFrame()
features["user_id"] = orders["user_id"]
features["last_order_date"] = orders["last_order_date"]
features["first_order_date"] = orders["first_order_date"]
features["order_count"] = orders["order_count"]
features["avg_order_value"] = orders["avg_order_value"]
features["avg_review_stars"] = orders["avg_review_stars"]
features["days_since_last_order"] = (today - features["last_order_date"]).dt.days
features["tenure_days"] = (today - features["first_order_date"]).dt.days
features["order_velocity"] = (features["order_count"] /(features["tenure_days"] + 1))

In [13]:
baseline = features.copy()
production = features.tail(7).copy()

In [14]:
numeric_cols = production.select_dtypes(include=np.number).columns
print("drift")
for col in numeric_cols:
    statistic, pvalue = ks_2samp(baseline[col],production[col])
    if pvalue < 0.05:
        status = "🔴 DRIFT"
    else:
        status = "🟢 PASS"
    print(f"{col:25}  p={pvalue:.4f}   {status}")

drift
user_id                    p=0.0000   🔴 DRIFT
order_count                p=1.0000   🟢 PASS
avg_order_value            p=nan   🟢 PASS
avg_review_stars           p=nan   🟢 PASS
days_since_last_order      p=nan   🟢 PASS
tenure_days                p=nan   🟢 PASS
order_velocity             p=nan   🟢 PASS
